<p style='text-align: center;'>

## <p style='text-align: center;'> **This is the Notebook for the CIFO Project**

### <p style='text-align: center;'> **Group: C37 Vestigial**




In [1]:
# Numerical arrays and vectorized operations used for triangle genomes and RMSE calculations.
import numpy as np

# Plotting library used to show final images and convergence curves inside the notebook.
import matplotlib.pyplot as plt

# Random utilities used by the genetic algorithm: initialization, selection, and mutation.
from random import randint, choices, sample, random

# OpenCV is used to load images, draw filled triangles, and save generated outputs.
import cv2

# ABC/abstractmethod are used to define a generic optimization solution template.
from abc import ABC, abstractmethod

# deepcopy prevents offspring/elites from accidentally sharing the same object in memory as parents.
from copy import deepcopy

In [2]:
class Solution(ABC):
    """Generic abstract class for an optimization solution."""

    def __init__(self, repr=None):
        # If no representation is provided, create a random one using the subclass method.
        if repr == None:
            repr = self.random_initial_representation()

        # Store the representation/genome of the solution.
        self.repr = repr

    def __repr__(self):
        # Defines how the solution is displayed when printed.
        return str(self.repr)

    @abstractmethod
    def fitness(self):
        # Every specific solution must define how quality/fitness is calculated.
        pass

    @abstractmethod
    def random_initial_representation(self):
        # Every specific solution must define how a random starting solution is created.
        pass

## <center> Section Containing the Algorithms to test for the Optimization Problem <center>

##### These Include:
- Hill Climbing
- Simulated Annealing
- Random Search

### <center> **Hill Climbing Algorithm** <center>

In [3]:
def hill_climbing(initial_solution, maximization=False, max_iter=9999, verbose=False):
    """Generic Hill Climbing algorithm kept from class material."""

    # Start from the provided initial solution.
    current = initial_solution

    # Controls whether the algorithm continues searching.
    improved = True

    # Counts iterations and stores the sequence of visited solutions.
    iter = 1
    seqsols = [current]

    # Continue while a better neighbor is found.
    while improved:
        if verbose:
            print(
                f"Current Solution: {current.repr} ({current}) with fitness {current.fitness()}"
            )

        improved = False

        # Gets the best neighbor of the current solution.
        # NOTE: VermeerSolution would need get_bestneighbor() implemented for this to work on the painting problem.
        bestneighb = current.get_bestneighbor()

        # For maximization problems, replace current if neighbor has higher/equal fitness.
        if maximization and bestneighb.fitness() >= current.fitness():
            current = deepcopy(bestneighb)
            improved = True

        # For minimization problems, replace current if neighbor has lower/equal fitness.
        elif not maximization and bestneighb.fitness() <= current.fitness():
            current = deepcopy(bestneighb)
            improved = True

        # Store only accepted improvements.
        if improved:
            seqsols.append(current)

        # Stop if maximum number of iterations is reached.
        iter += 1
        if iter == max_iter:
            break

    if verbose:
        print(
            f"Discarded best neighbor: {bestneighb.repr} ({bestneighb}) with fitness {bestneighb.fitness()}"
        )
        print(
            f"Best visited: {current.repr} ({current}) with fitness {current.fitness()}"
        )

    return current, seqsols

### <center> **Simulated Annealing Algorithm** <center>

Create generic Solution class as seen in Labs with methods fitness and random initial representation is repr is None

In [4]:
def simulated_annealing(
    initial_solution: Solution,
    C: float,
    L: int,
    H: float,
    maximization: bool = True,
    max_iter: int = 10,
    verbose: bool = False,
):
    """
    Generic Simulated Annealing implementation kept from class material.

    NOTE: VermeerSolution would need get_random_neighbor() implemented for this to work on the painting problem.
    """

    # Start from the provided initial solution and calculate its fitness.
    current_solution = initial_solution
    current_fitness = current_solution.fitness()

    # Outer-loop counter and list of accepted solutions.
    iter = 1
    history = []

    if verbose:
        print(
            f"Initial Solution {current_solution.repr} with fitness {current_fitness}"
        )

    # Outer loop: gradually cools the temperature/control parameter C.
    while iter <= max_iter:

        # Inner loop: test L random neighbors at the current temperature.
        for _ in range(L):
            random_neighbor = current_solution.get_random_neighbor()
            neighbor_fitness = random_neighbor.fitness()

            if verbose:
                print(
                    f"Random neighbor {random_neighbor} with fitness {neighbor_fitness}"
                )

            # Accept neighbor immediately if it improves the objective.
            if (maximization and (neighbor_fitness >= current_fitness)) or (
                not maximization and (neighbor_fitness <= current_fitness)
            ):
                current_solution = deepcopy(random_neighbor)
                current_fitness = current_solution.fitness()

                if verbose:
                    print(f"Neighbor is better, current solution is replaced")

            else:
                # Simulated Annealing may accept worse solutions to escape local optima.
                random_float = random()

                # Probability of accepting a worse solution decreases as C decreases.
                p = np.exp(-abs(current_fitness - neighbor_fitness) / C)

                if verbose:
                    print(f"Probability of accepting worse event: {p}")

                # Accept the worse neighbor if the random draw is below p.
                if random_float < p:
                    current_solution = deepcopy(random_neighbor)
                    current_fitness = current_solution.fitness()

                    if verbose:
                        print(f"Neighbor is worse and was accepted")
                else:
                    if verbose:
                        print(f"Neighbor is worse and was not accepted")

            # Store the current solution after each neighbor test.
            history.append(current_solution)

            if verbose:
                print(
                    f"History was appended with {current_solution} and fitness {current_fitness}"
                )

        # Cool down the temperature/control parameter.
        C = C / H

        if verbose:
            print(f"Decreased C, new value: {C}")

        iter += 1

    if verbose:
        print(
            f"Best solution found: {current_solution.repr} with fitness {current_fitness}"
        )

    return current_solution, history

In [5]:
class VermeerSolution(Solution):
    """A candidate painting made from 100 colored triangles."""

    def __init__(self, target_image, max_allowed_area, size_weight, repr=None):
        # Store the original image that the generated triangle image will be compared against.
        self.target_image = target_image

        # Project canvas size: 300x400 pixels.
        self.width = 300
        self.height = 400

        # Project requirement: use 100 triangles.
        self.num_triangles = 100

        # Attributes max_allowed_area and size_weight are created to control the size of triangles
        # and penalize triangles for growing a lot in order to avoid 'blobbing' of triangles
        self.max_allowed_area = max_allowed_area
        self.size_weight = size_weight

        # Calls Solution.__init__, which either stores a provided repr or creates a random one.
        super().__init__(repr=repr)

    def random_initial_representation(self):
        """
        Generates a random genome of 100 small triangles.
        Each row has: [x1, y1, x2, y2, x3, y3, r, g, b]
        """

        # Matrix with one row per triangle and 9 values per triangle.
        random_repr = np.zeros((self.num_triangles, 9), dtype=int)

        for i in range(self.num_triangles):
            # Pick a random center points in the canvas for the triangle
            cx = randint(0, self.width)
            cy = randint(0, self.height)

            # Define an max radius for the triangles
            radius = 40
            # Generate 3 random x-coordinates and 3 random y-coordinates within the range of the defined radius
            # This allows to control the size of the randomly intialized traingles (can start with small triangles)
            x1 = np.clip(cx + randint(-radius, radius), 0, self.width)
            y1 = np.clip(cy + randint(-radius, radius), 0, self.height)
            x2 = np.clip(cx + randint(-radius, radius), 0, self.width)
            y2 = np.clip(cy + randint(-radius, radius), 0, self.height)
            x3 = np.clip(cx + randint(-radius, radius), 0, self.width)
            y3 = np.clip(cy + randint(-radius, radius), 0, self.height)
            # Generate one random RGB color for the whole triangle.
            r, g, b = [randint(0, 255) for _ in range(3)]
            # Store the triangle's vertices and color in the genome.
            random_repr[i] = [x1, y1, x2, y2, x3, y3, r, g, b]

        return random_repr

    def render_canvas(self):
        """Converts the 100-triangle genome into an actual image matrix."""

        # Start from a black image with shape: height x width x RGB channels.
        canvas = np.zeros((self.height, self.width, 3), dtype=np.uint8)

        for gene in self.repr:
            # Extract the three triangle vertices in OpenCV's expected format.
            pts = np.array(
                [[gene[0], gene[1]], [gene[2], gene[3]], [gene[4], gene[5]]], np.int32
            )

            # Reshape points so cv2.fillPoly can draw the polygon.
            pts = pts.reshape((-1, 1, 2))

            # Extract the triangle color.
            # NOTE: The loaded OpenCV image is BGR, but the project logic treats these as channel values consistently.
            color = (int(gene[6]), int(gene[7]), int(gene[8]))

            # Draw a filled triangle on the canvas.
            cv2.fillPoly(canvas, [pts], color)

        return canvas

    def fitness(self):
        """Calculates RMSE between the target image and the generated triangle image.
        Applies a small penalty to overlapping triangles in order to try and resolve the problem with facial features
        """

        # Render this individual's genome into an image.
        generated_image = self.render_canvas()

        # Convert to float before subtracting, avoiding uint8 overflow/underflow errors.
        target_float = self.target_image.astype(np.float32)
        gen_float = generated_image.astype(np.float32)

        # Pixel-by-pixel difference between target and generated image.
        diff = target_float - gen_float
        # RMSE = sqrt(mean(square(error))). Lower RMSE means a better image.
        sq_diff = np.square(diff)
        mse = np.mean(sq_diff)
        rmse = np.sqrt(mse)

        # 2. The Size/Overlap Soft Penalty
        # Extract all X and Y coordinates to calculate area using the Shoelace Formula
        x1, y1 = self.repr[:, 0], self.repr[:, 1]
        x2, y2 = self.repr[:, 2], self.repr[:, 3]
        x3, y3 = self.repr[:, 4], self.repr[:, 5]

        # Calculate area of all 100 triangles at once
        areas = 0.5 * np.abs(x1 * (y2 - y3) + x2 * (y3 - y1) + x3 * (y1 - y2))

        # Define a maximum acceptable area (e.g., a 40x40 pixel square is 1600 area)
        max_allowed_area = self.max_allowed_area

        # Find triangles that exceed this area
        excess_area = np.maximum(0, areas - max_allowed_area)

        # Create a penalty proportional to the excess size
        # The weight (0.001) determines how strict the GA is about large triangles.
        # Higher = strictly small triangles. Lower = some big triangles allowed.
        size_weight = self.size_weight
        size_penalty = np.sum(excess_area) * size_weight

        # The final fitness is the visual error PLUS the geometric penalty
        return rmse + size_penalty

In [6]:
# Path to João's local copy of the target image.
joao_path = r"C:\Users\joaoa\Desktop\CIFO\data\girl_pearl_earing.png"
goncalo_path = r"C:\CIfO\Project\data\girl_pearl_earing.png"
# Load the original image using OpenCV.
original_image = cv2.imread(goncalo_path)

# Fail early with a clear error if the path is wrong or the image cannot be read.
if original_image is None:
    raise FileNotFoundError(f"OpenCV could not find the image at: {joao_path}")

# Create one random triangle painting to test the VermeerSolution class.
first_painting = VermeerSolution(
    target_image=original_image, max_allowed_area=1500, size_weight=0.00001
)

# Print its genome and RMSE fitness to confirm the representation and fitness function work.
print(f"{first_painting.repr} has fitness {first_painting.fitness()}")

[[ 68 115  33  92  67  73  48 244 183]
 [ 89 249  94 283  84 259 163 196 176]
 [ 48 137  16 103  52 104 239  29  69]
 [179 398 120 375 180 400 252  38 224]
 [156 220  95 186 162 234 241 246 108]
 [ 37 307  48 286  32 315 205 242 193]
 [131 374 130 351 169 365  13  24  39]
 [120 117 111 128 118 121 252 120 168]
 [107 243 128 241 132 274 229 238  38]
 [158 347 130 311 168 300 119 172 219]
 [263 374 274 384 263 400 213 134 234]
 [145 214 126 219 135 215 195 225  66]
 [194 189 218 227 226 219  31  56 199]
 [186 228 184 193 131 229  75 197  62]
 [  9 289  27 312  34 259 124 183  24]
 [ 66  44  67  33  66   8  46  69 249]
 [100  67  70  44  76  52 124   3  88]
 [ 65 349   5 400  10 347  42  53 143]
 [142 400 131 359 122 400  57 100 251]
 [211 211 193 211 226 190  44 223 223]
 [  9 312   0 336  29 320 244  65 216]
 [246 103 213 151 240 164 222 142 110]
 [ 56 364  20 355  64 361 100  16 152]
 [169   0 155   0 217   0 238   9 140]
 [212 128 265 126 207 113 121 103 143]
 [  0  76  67  82   0  72

Now to test out if the initial class functions in producing an image and comparing it to the original painting

In [ ]:
"""gonçalo_path = 'C:\CIfO\Project\data\girl_pearl_earing.png'

original_vermeer = cv2.imread(r'C:\CIfO\Project\data\girl_pearl_earing.png')

my_first_painting = VermeerSolution(target_image=original_vermeer)

score = round(my_first_painting.fitness(), 2)

print(f'The RMSE Score of random painting is: {score}')"""

The RMSE Score of random painting is: 90.41000366210938


Now I will create my GA class to perform optimization of my solution

**Notes**:
The probability of mutation should be 1 over the length of the representation (string)

In [7]:
class VermeerGA:
    """First Genetic Algorithm implementation for evolving triangle paintings."""

    def __init__(
        self,
        target_image,
        pop_size=50,
        generations=1000,
        mutation_rate=0.05,
        tournament_size=3,
        use_crossover=True,
        use_mutation=True,
    ):
        # Image that every generated painting will be compared against.
        self.target_image = target_image

        # Number of individuals in each generation.
        self.pop_size = pop_size

        # Number of evolutionary iterations.
        self.generations = generations

        # Probability of mutating each triangle.
        self.mutation_rate = mutation_rate

        # Number of individuals sampled in tournament selection.
        self.tournament_size = tournament_size

        # These switches are useful for Challenge 3 experiments.
        self.use_crossover = use_crossover
        self.use_mutation = use_mutation

        # Initial population: random triangle paintings.
        self.population = [
            VermeerSolution(target_image=self.target_image)
            for _ in range(self.pop_size)
        ]

    def evaluate_population(self):
        """Evaluate all individuals and sort population from best to worst RMSE."""

        # Recalculate fitness for every individual.
        # This avoids stale fitness values after mutation/crossover.
        for ind in self.population:
            ind.fitness_score = ind.fitness()

        # Since RMSE is minimized, the best individual is at index 0.
        self.population.sort(key=lambda ind: ind.fitness_score)

    def tournament_selection(self):
        """Picks K random individuals and returns the one with the lowest RMSE."""

        # Randomly sample candidates from the current population.
        tournament = sample(self.population, self.tournament_size)

        # Return the best candidate among the sampled individuals.
        winner = sorted(tournament, key=lambda ind: ind.fitness_score)[0]
        return winner

    def crossover(self, parent1, parent2):
        """Single-point crossover with a random split point."""

        # Create an empty/random child first; its representation will be overwritten below.
        child = VermeerSolution(target_image=self.target_image)

        # Choose a split point between triangle 1 and triangle 99.
        split_point = np.random.randint(1, parent1.num_triangles)

        # Child receives the first part from parent1 and the second part from parent2.
        child.repr[:split_point] = parent1.repr[:split_point].copy()
        child.repr[split_point:] = parent2.repr[split_point:].copy()

        return child

    def mutate(self, child):
        """Randomly nudges vertices and colors of the triangles."""

        for i in range(child.num_triangles):
            # Each triangle has a chance of being mutated.
            if random() < self.mutation_rate:

                # Slightly move the 3 vertices: [x1, y1, x2, y2, x3, y3].
                child.repr[i][0:6] += np.random.randint(-15, 16, 6)

                # Slightly change the RGB color values.
                child.repr[i][6:9] += np.random.randint(-20, 21, 3)

                # Keep x-coordinates inside [0, width].
                child.repr[i][0] = np.clip(child.repr[i][0], 0, child.width)
                child.repr[i][2] = np.clip(child.repr[i][2], 0, child.width)
                child.repr[i][4] = np.clip(child.repr[i][4], 0, child.width)

                # Keep y-coordinates inside [0, height].
                child.repr[i][1] = np.clip(child.repr[i][1], 0, child.height)
                child.repr[i][3] = np.clip(child.repr[i][3], 0, child.height)
                child.repr[i][5] = np.clip(child.repr[i][5], 0, child.height)

                # Keep RGB values inside [0, 255].
                child.repr[i][6:9] = np.clip(child.repr[i][6:9], 0, 255)

        return child

    def run(self):
        """Main evolutionary loop."""

        print(f"Starting Evolution for {self.generations} generations...")

        # Stores the best RMSE from each generation for convergence plots.
        history = []

        # Evaluate initial population before selection is used.
        self.evaluate_population()

        for gen in range(self.generations):
            # Because the population is sorted, index 0 is the current best individual.
            best_current_solution = self.population[0]

            # Save current best fitness for later plotting/reporting.
            history.append(best_current_solution.fitness_score)

            # Print progress every 10 generations.
            if gen % 10 == 0:
                print(
                    f"Generation {gen} | Best RMSE: {best_current_solution.fitness_score:.2f}"
                )

            # Build the next population from scratch.
            next_generation = []

            # Elitism: keep the best painting unchanged so the best score cannot be lost.
            next_generation.append(deepcopy(best_current_solution))

            # Fill the rest of the next generation with offspring.
            while len(next_generation) < self.pop_size:

                # Select two parents using tournament selection.
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()

                # Either create a crossover child or copy one parent directly.
                if self.use_crossover:
                    child = self.crossover(p1, p2)
                else:
                    child = deepcopy(p1)

                # Apply mutation unless disabled for experiments.
                if self.use_mutation:
                    child = self.mutate(child)

                # Add the new child to the next generation.
                next_generation.append(child)

            # Replace old population with the newly created one.
            self.population = next_generation

            # Recalculate and sort after crossover/mutation.
            self.evaluate_population()

        print("Evolution Complete!")

        # Final safety evaluation/sort before returning the best individual.
        self.evaluate_population()
        return self.population[0], history

### New Implementation of the GA with upgrades Mutation, Crossover and Elitism

In [8]:
class VermeerGA2:
    """Improved Genetic Algorithm with uniform crossover, richer mutation, elitism, and optional saving."""

    def __init__(
        self,
        target_image,
        pop_size=200,
        generations=1000,
        pc=0.85,
        pm=0.03,
        elitism_size=1,
        tournament_size=3,
        use_crossover=True,
        use_mutation=True,
        save_every=None,
        output_dir=None,
        max_allowed_area=1500,
        size_weight=0.00001,
    ):
        # Target image used by the RMSE fitness function.
        self.target_image = target_image

        # Population and run-length parameters.
        self.pop_size = pop_size
        self.generations = generations

        # Crossover probability and mutation probability.
        self.pc = pc
        self.pm = pm

        # Number of best individuals copied unchanged into the next generation.
        self.elitism_size = elitism_size

        # Number of candidates used in tournament selection.
        self.tournament_size = tournament_size

        # Switches used for Challenge 3 ablation experiments.
        self.use_crossover = use_crossover
        self.use_mutation = use_mutation

        # Optional controls for saving intermediate images.
        self.save_every = save_every
        self.output_dir = output_dir

        # Size penalty hyperparameters
        self.max_allowed_area = max_allowed_area
        self.size_weight = size_weight

        # Initial random population.
        self.population = [
            VermeerSolution(
                target_image=self.target_image,
                max_allowed_area=self.max_allowed_area,
                size_weight=self.size_weight,
            )
            for _ in range(self.pop_size)
        ]

    def evaluate_population(self):
        """Evaluate all individuals and sort population from best to worst RMSE."""

        # Always recalculate fitness after reproduction to avoid stale fitness values.
        for ind in self.population:
            ind.fitness_score = ind.fitness()

        # Lower RMSE is better, so ascending order places the best individual first.
        self.population.sort(key=lambda ind: ind.fitness_score)

    def tournament_selection(self):
        """Picks K random individuals and returns the one with the lowest RMSE."""

        # Randomly select tournament_size candidates.
        tournament = sample(self.population, self.tournament_size)

        # The candidate with lowest RMSE wins the tournament.
        winner = sorted(tournament, key=lambda ind: ind.fitness_score)[0]
        return winner

    def crossover(self, parent1, parent2):
        """Uniform crossover: each triangle is inherited from either Parent 1 or Parent 2."""

        # Create a child object; then replace its genome using parent genes.
        child = VermeerSolution(
            target_image=self.target_image,
            max_allowed_area=self.max_allowed_area,
            size_weight=self.size_weight,
        )

        # Mask has one 0/1 value per triangle.
        # 1 means take triangle from parent1; 0 means take triangle from parent2.
        mask = np.random.randint(0, 2, size=(child.num_triangles, 1))

        # np.where applies the mask triangle by triangle.
        child.repr = np.where(mask, parent1.repr, parent2.repr).copy()

        return child

    def mutate(self, child):
        """Randomly mutates triangle coordinates, colors, or drawing order."""

        for i in range(child.num_triangles):
            # Each triangle has probability pm of being mutated.
            if random() < self.pm:

                # Decide which mutation operator to apply.
                mutation_type = random()

                if mutation_type < 0.80:
                    # 80% of mutations are small nudges to preserve useful structures.
                    child.repr[i][0:6] += np.random.randint(-10, 11, 6)  # coordinates
                    child.repr[i][6:9] += np.random.randint(-15, 16, 3)  # color

                    # Keep x-coordinates inside the image width.
                    child.repr[i][0] = np.clip(child.repr[i][0], 0, child.width)
                    child.repr[i][2] = np.clip(child.repr[i][2], 0, child.width)
                    child.repr[i][4] = np.clip(child.repr[i][4], 0, child.width)

                    # Keep y-coordinates inside the image height.
                    child.repr[i][1] = np.clip(child.repr[i][1], 0, child.height)
                    child.repr[i][3] = np.clip(child.repr[i][3], 0, child.height)
                    child.repr[i][5] = np.clip(child.repr[i][5], 0, child.height)

                    # Keep color channels valid.
                    child.repr[i][6:9] = np.clip(child.repr[i][6:9], 0, 255)

                elif mutation_type < 0.90:
                    # 10% of mutations fully replace one triangle, helping exploration.
                    child.repr[i] = [
                        randint(0, child.width),
                        randint(0, child.height),
                        randint(0, child.width),
                        randint(0, child.height),
                        randint(0, child.width),
                        randint(0, child.height),
                        randint(0, 255),
                        randint(0, 255),
                        randint(0, 255),
                    ]

                else:
                    # 10% of mutations swap triangle drawing order.
                    # This matters because later triangles can cover earlier triangles.
                    swap_idx = randint(0, child.num_triangles - 1)
                    child.repr[[i, swap_idx]] = child.repr[[swap_idx, i]]

        return child

    def maybe_save_generation(self, gen):
        """Optionally save the current best image every N generations."""

        # Do nothing if saving was not configured.
        if self.save_every is None or self.output_dir is None:
            return

        # Only save at selected intervals.
        if gen % self.save_every != 0:
            return

        # Create output folder if it does not already exist.
        import os

        os.makedirs(self.output_dir, exist_ok=True)

        # Save the current best image; population[0] is best because population is sorted.
        image = self.population[0].render_canvas()
        cv2.imwrite(f"{self.output_dir}/gen_{gen}.png", image)

    def run(self):
        """Main evolutionary loop."""

        print(f"Starting Evolution for {self.generations} generations...")

        # Stores the best RMSE at each generation for convergence analysis.
        history = []

        # Evaluate initial population before tournament selection starts.
        self.evaluate_population()

        for gen in range(self.generations):
            # Best individual is always first after sorting.
            best_current_solution = self.population[0]

            # Store best RMSE for this generation.
            history.append(best_current_solution.fitness_score)

            # Print progress and optionally save image snapshots.
            if gen % 50 == 0:
                print(
                    f"Generation {gen} | Best RMSE: {best_current_solution.fitness_score:.2f}"
                )
                self.maybe_save_generation(gen)

            # Elitism: copy the best k individuals directly to the next generation.
            next_generation = [
                deepcopy(ind) for ind in self.population[: self.elitism_size]
            ]

            # Generate children until the next population is full.
            while len(next_generation) < self.pop_size:

                # Select the first parent.
                p1 = self.tournament_selection()

                # With probability pc, create a child through crossover.
                # If crossover is disabled, clone p1 instead.
                if self.use_crossover and random() < self.pc:
                    p2 = self.tournament_selection()
                    child = self.crossover(p1, p2)
                else:
                    child = deepcopy(p1)

                # Apply mutation unless disabled for Challenge 3 experiments.
                if self.use_mutation:
                    child = self.mutate(child)

                # Add the new individual to the next generation.
                next_generation.append(child)

            # Replace old population with new generation.
            self.population = next_generation

            # Evaluate and sort all new individuals before the next generation.
            self.evaluate_population()

        # Final evaluation/sort for safety before returning.
        self.evaluate_population()

        print(f"Evolution Complete! Final RMSE: {self.population[0].fitness_score:.2f}")
        return self.population[0], history

In [9]:
import os
import cv2
import matplotlib.pyplot as plt

if __name__ == "__main__":

    # 1. Define Base Directory
    base_results_dir = r"C:\CIfO\Project\data\results"

    # Create base directory if it does not exist
    os.makedirs(base_results_dir, exist_ok=True)

    # 2. Create New Folder to be used for each run
    existing_runs = [
        folder
        for folder in os.listdir(base_results_dir)
        # Folders will be organized by the Nº of Run, so first folder will be the first trial Run then subsequent runs will be 2, 3, 4 ...
        if folder.startswith("Run")
    ]
    # Create list to track the number of runs performed
    run_numbers = []

    for folder in existing_runs:
        try:
            number = int(folder.replace("Run", ""))
            run_numbers.append(number)
        except:
            pass
    # Looks at the last run performed and adds the next run max(run_numbers, default=0) + 1
    next_run_number = max(run_numbers, default=0) + 1

    run_folder = os.path.join(base_results_dir, f"Run{next_run_number}")

    os.makedirs(run_folder, exist_ok=True)

    print(f"Results will be saved in: {run_folder}")

    # 3. Define Target image path
    goncalo_target_path = r"C:\CIfO\Project\data\girl_pearl_earing.png"

    # 4. Load Target Image

    target = cv2.imread(goncalo_target_path)

    if target is None:
        raise FileNotFoundError(
            f"OpenCV could not find the image at: {goncalo_target_path}"
        )

    # 5. Define Hyperparameters (These are changed in each different run for optimization, however they are saved at each run)

    pop_size = 200
    generations = 8000
    pc = 0.85
    pm = 0.05
    elitism_size = 10
    tournament_size = 4
    save_every = 500
    max_allowed_area = 1600
    size_weight = 0.000001
        
        
    # 6. Initialize GA
    ga = VermeerGA2(
        target_image=target,
        pop_size=pop_size,
        generations=generations,
        pc=pc,
        pm=pm,
        elitism_size=elitism_size,
        tournament_size=tournament_size,
        save_every=save_every,
        output_dir=run_folder,
        max_allowed_area=max_allowed_area,
        size_weight=size_weight,
    )

    best_painting, fitness_history = ga.run()
    
    # 7. Creates a text file (hyperparameters.txt) for each run folder to use as a reference for optimization
    hyperparameter_path = os.path.join(run_folder, "hyperparameters.txt")

    with open(hyperparameter_path, "w") as file:

        file.write(f"GA2 Hyperparameters for Run Number: {next_run_number}\n\n")
        write_notes = str(
            input("Do you wish to add notes to this run (select yes/no)").lower()
        )
        if write_notes == "yes":
            file.write(input("Notes for this run: "))
            file.write(f"\n")
        else:
            pass
        file.write(f"Population Size: {pop_size}\n")
        file.write(f"Generations: {generations}\n")
        file.write(f"Crossover Probability (pc): {pc}\n")
        file.write(f"Mutation Probability (pm): {pm}\n")
        file.write(f"Elitism Size: {elitism_size}\n")
        file.write(f"Tournament Size: {tournament_size}\n")
        file.write(f"Save Every: {save_every}\n")
        file.write(f"Max Allowed Area: {max_allowed_area}\n")
        file.write(f"Weight size: {size_weight:.5f}\n")
        file.write(f"Target Image: {goncalo_target_path}\n")
        file.write(f"Final RMSE Score: {best_painting.fitness_score:.4f}")

    final_image = best_painting.render_canvas()

    # 10. Save Final Image
    final_image_path = os.path.join(run_folder, "final_result.png")
    success = cv2.imwrite(final_image_path, final_image)
    if success:
        print(f"Final image saved at:\n{final_image_path}")
    else:
        print("Failed to save final image.")

    # 11. Display Final Image
    plt.figure(figsize=(6, 8))
    plt.imshow(cv2.cvtColor(final_image, cv2.COLOR_BGR2RGB))
    plt.title("Best Vermeer Approximation - GA2")
    plt.axis("off")
    plt.show()

    # 12. Plot Convergence Curve
    plt.figure(figsize=(10, 5))
    plt.plot(fitness_history)
    plt.xlabel("Generation")
    plt.ylabel("RMSE")
    plt.title("GA2 Convergence")
    plt.grid(True)
    convergence_plot_path = os.path.join(run_folder, "convergence_plot.png")
    plt.savefig(convergence_plot_path, dpi=300, bbox_inches="tight")
    plt.show()

Results will be saved in: C:\CIfO\Project\data\results\Run6
Starting Evolution for 8000 generations...
Generation 0 | Best RMSE: 99.37
Generation 50 | Best RMSE: 51.41
Generation 100 | Best RMSE: 44.91
Generation 150 | Best RMSE: 42.17
Generation 200 | Best RMSE: 40.22
Generation 250 | Best RMSE: 39.12
Generation 300 | Best RMSE: 38.43
Generation 350 | Best RMSE: 37.95
Generation 400 | Best RMSE: 37.21
Generation 450 | Best RMSE: 36.99
Generation 500 | Best RMSE: 36.58
Generation 550 | Best RMSE: 36.32
Generation 600 | Best RMSE: 36.11
Generation 650 | Best RMSE: 35.90
Generation 700 | Best RMSE: 35.71
Generation 750 | Best RMSE: 35.58
Generation 800 | Best RMSE: 35.34
Generation 850 | Best RMSE: 35.12
Generation 900 | Best RMSE: 34.98
Generation 950 | Best RMSE: 34.90
Generation 1000 | Best RMSE: 34.82
Generation 1050 | Best RMSE: 34.81
Generation 1100 | Best RMSE: 34.80
Generation 1150 | Best RMSE: 34.74
Generation 1200 | Best RMSE: 34.68
Generation 1250 | Best RMSE: 34.66
Generation

KeyboardInterrupt: 

### Challenge 3: GA component comparison

This section compares the contribution of crossover, mutation, and selection pressure using the same target image and comparable parameters.


In [ ]:
def run_challenge3_experiments(
    target, pop_size=100, generations=500, pc=0.85, pm=0.03, elitism_size=3
):
    """
    Runs fair GA component experiments for Challenge 3.

    Fair comparison principle:
    - Keep population size, generations, pc, pm and elitism fixed.
    - Change only the component being tested.
    """

    # Each configuration changes one GA component relative to the baseline.
    experiment_configs = {
        "Baseline": {
            "use_crossover": True,  # Normal GA: crossover enabled.
            "use_mutation": True,  # Normal GA: mutation enabled.
            "tournament_size": 3,  # Standard selection pressure.
        },
        "No crossover": {
            "use_crossover": False,  # Tests how much crossover contributes.
            "use_mutation": True,
            "tournament_size": 3,
        },
        "No mutation": {
            "use_crossover": True,
            "use_mutation": False,  # Tests how much mutation contributes.
            "tournament_size": 3,
        },
        "Stronger selection": {
            "use_crossover": True,
            "use_mutation": True,
            "tournament_size": 6,  # Tests the impact of stronger tournament selection. (Increases selection pressure/
            # Lower fitness individuals are less likely to be selected)
        },
    }

    # Stores best solutions, histories, and final RMSE values for each experiment.
    results = {}

    # Run every configuration using the same base parameters.
    for name, config in experiment_configs.items():
        print(f"\nRunning experiment: {name}")

        # Create a GA using the current experiment's component settings.
        ga = VermeerGA2(
            target_image=target,
            pop_size=pop_size,
            generations=generations,
            pc=pc,
            pm=pm,
            elitism_size=elitism_size,
            tournament_size=config["tournament_size"],
            use_crossover=config["use_crossover"],
            use_mutation=config["use_mutation"],
        )

        # Run the experiment.
        best_solution, history = ga.run()

        # Save all data needed for comparison and plots.
        results[name] = {
            "best_solution": best_solution,
            "history": history,
            "final_rmse": history[-1],
        }

    return results


def plot_challenge3_results(results):
    """Plots convergence curves for all Challenge 3 experiments."""

    # Create one convergence plot comparing all configurations.
    plt.figure(figsize=(10, 6))

    # Plot each experiment's RMSE history.
    for name, result in results.items():
        plt.plot(
            result["history"], label=f"{name} - final RMSE {result['final_rmse']:.2f}"
        )

    # Add report-ready labels and legend.
    plt.xlabel("Generation")
    plt.ylabel("RMSE")
    plt.title("Challenge 3: Contribution of GA Components")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# Keep False by default so the notebook does not run long experiments accidentally.
# Set to True only when you want to run Challenge 3 experiments.
run_config_experiments = True

if run_config_experiments:
    # Load the same target image used in the main GA run.
    user = str(input("Select user (joao/goncalo): ").lower())
    if user == "joao":
        target_path = r"C:\Users\joaoa\Desktop\CIFO\data\girl_pearl_earing.png"
    elif user == "goncalo":
        target_path = r"C:\CIfO\Project\data\girl_pearl_earing.png"
    target = cv2.imread(goncalo_target_path)

    # Stop if the target image cannot be loaded.
    if target is None:
        raise FileNotFoundError(
            f"OpenCV could not find the image at: {goncalo_target_path}"
        )

    # Run Challenge 3 with smaller values first because this can take a long time.
    challenge3_results = run_challenge3_experiments(
        target, pop_size=100, generations=500, pc=0.85, pm=0.03, elitism_size=3
    )

    # Plot the convergence curves for comparison.
    plot_challenge3_results(challenge3_results)


Running experiment: Baseline


AttributeError: 'VermeerGA2' object has no attribute 'size_weight'